In [54]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
load_dotenv()

True

In [56]:
model = ChatOpenAI(model_name="gpt-4o-mini")
# model.invoke('hi')

In [57]:
# history = InMemoryChatMessageHistory()
# history.messages

In [58]:
# RunnableWithMessageHistory

# step 1 : defined your prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify complaints as: Billing / Technical / General.
Priority: High (financial impact > $100 or urgent) / Medium / Low.
Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])

# step 2 LCEL
chain = prompt | model | StrOutputParser()


In [59]:
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [60]:
# step 4:
email_assistant = RunnableWithMessageHistory(
            chain,
            get_session_history,
            input_messages_key="input",
            history_messages_key="history"

)

/Users/rahultiwari/Documents/wills_18th_july_batch/ai-env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [61]:
store

{}

In [62]:
cfg = {"configurable":{"session_id":"123rahul"}}

r1 = email_assistant.invoke({"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
                            config=cfg
)

In [64]:
print(r1)

Classification: Billing  
Priority: High  

Customer Details: John  
Invoice Number: #1042  
Issue: Charged $500 instead of $250 (financial impact > $100).


In [79]:
store['123rahul'].messages

[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classification: Billing  \nPriority: High  \n\nCustomer Details: John  \nInvoice Number: #1042  \nIssue: Charged $500 instead of $250 (financial impact > $100).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(con

In [78]:
# follow up 
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

print(r2)

John's complaint falls under the category of Billing.


In [69]:
store['123rahul'].messages

[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classification: Billing  \nPriority: High  \n\nCustomer Details: John  \nInvoice Number: #1042  \nIssue: Charged $500 instead of $250 (financial impact > $100).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [73]:
# follow up 

cfg = {"configurable":{"session_id":"123rahul"}}
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

In [71]:
store

{'123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Classification: Billing  \nPriority: High  \n\nCustomer Details: John  \nInvoice Number: #1042  \nIssue: Charged $500 instead of $250 (financial impact > $100).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'test': InMemoryChatMessageHistory(messages=[HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content='In order to classify the complaint, I need to know the specific 

In [74]:
r2

"John's complaint falls under the category of Billing."

In [39]:
store

{}

In [44]:
get_session_history('rahul123')

InMemoryChatMessageHistory(messages=[])

In [46]:
get_session_history('rahul125')

InMemoryChatMessageHistory(messages=[])

In [51]:
store['rahul123'].messages

[]

In [49]:
store

{'rahul123': InMemoryChatMessageHistory(messages=[]),
 'rahul124': InMemoryChatMessageHistory(messages=[]),
 'rahul125': InMemoryChatMessageHistory(messages=[])}

In [35]:
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a professional email support agent"),
    MessagesPlaceholder(variable_name="history"),  # slot for past messages
    ('human',"{input}"),

])

In [ ]:
# eg. 
past_messages = [
    HumanMessage(content="Billing complaint from John — invoice #1042, overcharged $250."),
    AIMessage(content="Classified: Billing — High Priority. SLA: 2 hours."),
    HumanMessage(content="what was the previous ticket classification category?"),
    AIMessage(content="The previous ticket classification category was 'Billing — High Priority' with an SLA of 2 hours.")
]

formatted = prompt.format_messages(history=past_messages,
    input="what was the previous ticket classification category."
)

formatted


[SystemMessage(content='You are a professional email support agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Billing complaint from John — invoice #1042, overcharged $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classified: Billing — High Priority. SLA: 2 hours.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what was the previous ticket classification category.', additional_kwargs={}, response_metadata={})]

In [37]:
print(model.invoke(formatted).content)

The previous ticket classification category was "Billing — High Priority."


In [22]:
prompt

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

In [6]:
history.add_user_message("Hello, how are you?")
history.add_ai_message("I'm good, thank you!")

In [7]:
history.messages

[HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm good, thank you!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [15]:
query = "Hi How are you"

response = model.invoke(query)

history.add_user_message(query)
history.add_ai_message(response)


# follow up questions
template = """ 
Previous conversation:
{history}
Current questions : {question}
"""

prompt = template.format(history=history.messages, question="What are you doing today?")


In [11]:
history.messages

[HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm good, thank you!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Hi How are you', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 11, 'total_tokens': 40, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f5d25cc737', 'id': 'chatcmpl-EIaPLrfd29nz5lYHnKMwHXzbXU3wN', 'service_tier': 'default', 'finis

In [16]:
prompt

' \nPrevious conversation:\n[HumanMessage(content=\'Hello, how are you?\', additional_kwargs={}, response_metadata={}), AIMessage(content="I\'m good, thank you!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content=\'Hi How are you\', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! I\'m just a program, so I don\'t have feelings, but I\'m here and ready to help you. How can I assist you today?", additional_kwargs={\'refusal\': None}, response_metadata={\'token_usage\': {\'completion_tokens\': 29, \'prompt_tokens\': 11, \'total_tokens\': 40, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cache_write_tokens\': None, \'cached_tokens\': 0}}, \'model_provider\': \'openai\', \'model_name\': \'gpt-4o-mini-2024-07-18\', \'system_fingerprint\': \'fp_f5d25cc737\', \'id\': 